## Import packages 

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

In [ ]:
# Implement this code in order to make any figures always use Times New Roman. Otherwise when you do it in the figures the matplotlib default can override it
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 12

# Set figure titles to be the same
mpl.rcParams['axes.titlesize'] = 20
mpl.rcParams['axes.titleweight'] = 'bold'
mpl.rcParams['axes.titlepad'] = 20
mpl.rcParams['font.family'] = 'Times New Roman'

## Set up base paths and paths to read in files

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
land_use = base_path / "Inputs/2013_landuse_LandCover.shp"
terrestrial_landcover = gpd.read_file(land_use)

In [ ]:
hydrobasins = base_path / "Processed_data/HydroBASINS_Level12_Clipped_Jamaica.shp"
hydrobasins = gpd.read_file(hydrobasins)
print(hydrobasins.crs)

In [ ]:
jamaica_boundary_path = base_path / "Inputs/Boundaries/jamaica.gpkg"
jamaica_boundary = gpd.read_file(jamaica_boundary_path)
print(f"Original Jamaica boundary CRS: {jamaica_boundary.crs}")

## Set up output folder 

In [ ]:
# Define the output folder path
output_folder = base_path / "Processed_data/existing_future_forest"

In [ ]:
map_forest_connectivity_figures_folder = base_path / "Results/Map_Figures/forest_connectivity"

In [ ]:
jamaica_metric_grid_crs = "EPSG:3448"

In [ ]:
# Reproject to Jamaica Metric Grid (EPSG:3448)
terrestrial_landcover = terrestrial_landcover.to_crs(jamaica_metric_grid_crs)
print("Landcover CRS:", terrestrial_landcover.crs)

## Define which classes count as "Existing_forest_plus_afforestable non-agricultural lands"

In [ ]:
existing_forest_and_afforestable_non_agri_classes = [
     'Open dry forest - Short',
     'Open dry forest - Tall (Woodland/Savanna)',
     'Disturbed broadleaved forest (Secondary Forest)',
     'Closed broadleaved forest (Primary Forest)',
     'Secondary Forest',
     'Fields and Secondary Forest',  # New mixed class
     'Bamboo and Secondary Forest',
     'Fields: Bare Land',
     'Quarry',
     'Bauxite Extraction',
]

## Filter existing_non agri afforestable

In [ ]:
existing_forest_and_afforestable_non_agri = terrestrial_landcover[terrestrial_landcover["Classify"].isin(existing_forest_and_afforestable_non_agri_classes)].copy()
print(f"Number of forest polygons: {len(existing_forest_and_afforestable_non_agri)}")



In [ ]:
# 3. Calculate geometry metrics for the existing_forest_and_afforestable_non_agri
existing_forest_and_afforestable_non_agri['area'] = existing_forest_and_afforestable_non_agri.geometry.area
existing_forest_and_afforestable_non_agri['perimeter'] = existing_forest_and_afforestable_non_agri.geometry.length

In [ ]:
print(existing_forest_and_afforestable_non_agri['area'] )

## Merge forests with catchments 

In [ ]:
existing_forest_and_afforestable_non_agri_in_basins = gpd.overlay(existing_forest_and_afforestable_non_agri, hydrobasins, how='intersection')

In [ ]:
# Calculate area in m² and add as a new column; convert to hectares later (1 ha = 10,000 m²)
existing_forest_and_afforestable_non_agri_in_basins["area_m2"] = existing_forest_and_afforestable_non_agri_in_basins.geometry.area

# Group by forest class to get total area for each class
area_by_class = existing_forest_and_afforestable_non_agri_in_basins.groupby("Classify")["area_m2"].sum()

# Total forest area across all classes
total_area = area_by_class.sum()

# Calculate percentage of total forest area for each class
percent_by_class = (area_by_class / total_area) * 100

In [ ]:
# Define a color mapping for each class (adjust hex codes as desired)
color_dict = {
    'Fields: Bare Land': '#fdae61',
    'Quarry': '#d7191c',
    'Bauxite Extraction': '#2b83ba',
    'Closed broadleaved forest (Primary Forest)': '#00441b',  # very dark green
    'Secondary Forest': '#2a924a',                             # moderately dark green
    'Disturbed broadleaved forest (Secondary Forest)': '#32CD32',  # noticeably lighter green
    'Open dry forest - Short': '#c7e9c0',                      # light green
    'Open dry forest - Tall (Woodland/Savanna)': '#A4C639',      # very light green
    'Fields and Secondary Forest': '#f0e442', 
    'Bamboo and Secondary Forest': '#9ACD32', # base color for agriculture part
}

In [ ]:
# Define a dictionary for the mixed classes and their descriptions (percentages)
mixed_classes = {
    'Fields and Secondary Forest': '75% Ag, 25% Forest',
    'Bamboo and Secondary Forest': '75% Bamboo, 25% Forest',
}


In [ ]:
# Set up the plot with common extents (using hydrobasins extent for context)
common_xlim = (593635.9271443005, 848114.2122774671)
common_ylim = (612896.835085367, 712816.0327676072)


In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

# Plot hydrobasins as a background layer
hydrobasins.plot(ax=ax, color="white", edgecolor="blue")

# Plot each agricultural class and build legend handles with area info
legend_handles = []
for cls, base_color in color_dict.items():
    subset = terrestrial_landcover[terrestrial_landcover["Classify"] == cls]
    if not subset.empty:
        if cls in mixed_classes:
            # For mixed classes, add a hatch pattern
            subset.plot(ax=ax, color=base_color, edgecolor="black", alpha=0.7,
                        hatch='//', zorder=101)
            description = mixed_classes[cls]
            total_area = area_by_class.get(cls, 0)
            total_area_ha = total_area / 10000  # Convert m² to ha
            label = f"{cls} ({description}, {total_area_ha:,.0f} ha)"
            patch = mpatches.Patch(facecolor=base_color, hatch='//', edgecolor="black", label=label)
            legend_handles.append(patch)
        else:
            subset.plot(ax=ax, color=base_color, edgecolor="black", alpha=0.7, zorder=101)
            total_area = area_by_class.get(cls, 0)
            total_area_ha = total_area / 10000  # Convert m² to ha
            label = f"{cls} ({total_area_ha:,.0f} ha)"
            legend_handles.append(mpatches.Patch(color=base_color, label=label))

# Optionally, add a legend entry for the hydrobasins background layer
bg_legend = Line2D([0], [0], color='blue', lw=2, label='Catchments')
legend_handles.insert(0, bg_legend)

# Define scale bar and north arrow functions (as before)
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Add the scale bar and north arrow to the map
add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
add_north_arrow(ax, location=(0.9, 0.85))

# Set axis limits, labels, and title
ax.set_xlim(common_xlim)
ax.set_ylim(common_ylim)
ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
ax.set_title("Existing forest and afforestable land: non-agricultural lands", fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

# Add the legend (positioned beneath the plot)
ax.legend(handles=legend_handles, title="Existing forest and afforestable non-agricultural classes", 
          loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=3, 
          frameon=False, fontsize=12, title_fontsize=14)

plt.tight_layout()
fig.savefig(map_forest_connectivity_figures_folder / "existing_forest_and_afforestable_non_agri_non_other_ecosystem_classes.png", dpi=300, bbox_inches="tight")
plt.show()

### Recalculate geometry metrics for the intersected polygons, since the intersection may change their sizes and edge lengths.

In [ ]:
existing_forest_and_afforestable_non_agri_in_basins['area'] = existing_forest_and_afforestable_non_agri_in_basins.geometry.area
existing_forest_and_afforestable_non_agri_in_basins['perimeter'] = existing_forest_and_afforestable_non_agri_in_basins.geometry.length

##### Area and edge 

In [ ]:
existing_forest_and_afforestable_non_agri_in_basins['area'] = existing_forest_and_afforestable_non_agri_in_basins.geometry.area
existing_forest_and_afforestable_non_agri_in_basins['perimeter'] = existing_forest_and_afforestable_non_agri_in_basins.geometry.length

##### Other statistics:
######   - patch_count: number of forest patch segments in the hydrobasin.
###### - avg_patch_size: average area of the patches.
###### - largest_patch_size: maximum patch area.
###### - total_edge_length: total length of patch edges.

In [ ]:
# --- Compute aggregated forest statistics from the intersection ---
existing_forest_plus_non_agri_stats = existing_forest_and_afforestable_non_agri_in_basins.groupby('HYBAS_ID').agg(
    patch_count=('area', 'count'),
    avg_patch_size=('area', 'mean'),
    largest_patch_size=('area', 'max'),
    total_edge_length=('perimeter', 'sum'),
    total_forest_area=('area', 'sum')
).reset_index()

# Convert forest areas from m² to hectares
existing_forest_plus_non_agri_stats['avg_patch_size_ha'] = existing_forest_plus_non_agri_stats['avg_patch_size'] / 10000
existing_forest_plus_non_agri_stats['largest_patch_size_ha'] = existing_forest_plus_non_agri_stats['largest_patch_size'] / 10000
existing_forest_plus_non_agri_stats['total_forest_area_ha'] = existing_forest_plus_non_agri_stats['total_forest_area'] / 10000

In [ ]:
# --- Merge aggregated stats with all hydrobasins ---
# Start with all hydrobasins (which has 103 rows) and merge forest_stats onto it.
existing_forest_plus_non_agri_basin_stats = hydrobasins[['HYBAS_ID', 'geometry']].copy()
existing_forest_plus_non_agri_basin_stats = existing_forest_plus_non_agri_basin_stats.merge(existing_forest_plus_non_agri_stats, on='HYBAS_ID', how='left')  # left join ensures all catchments are kept

# Calculate the catchment area (in m²) for each hydrobasin
existing_forest_plus_non_agri_basin_stats['catchment_area'] = existing_forest_plus_non_agri_basin_stats.geometry.area

# --- Fill missing values for catchments with no forest ---
# These NaNs occur in catchments with no intersecting forest features.
existing_forest_plus_non_agri_basin_stats['patch_count'] = existing_forest_plus_non_agri_basin_stats['patch_count'].fillna(0)
existing_forest_plus_non_agri_basin_stats['avg_patch_size'] = existing_forest_plus_non_agri_basin_stats['avg_patch_size'].fillna(0)
existing_forest_plus_non_agri_basin_stats['largest_patch_size'] = existing_forest_plus_non_agri_basin_stats['largest_patch_size'].fillna(0)
existing_forest_plus_non_agri_basin_stats['total_edge_length'] = existing_forest_plus_non_agri_basin_stats['total_edge_length'].fillna(0)
existing_forest_plus_non_agri_basin_stats['total_forest_area'] = existing_forest_plus_non_agri_basin_stats['total_forest_area'].fillna(0)
existing_forest_plus_non_agri_basin_stats['avg_patch_size_ha'] = existing_forest_plus_non_agri_basin_stats['avg_patch_size_ha'].fillna(0)
existing_forest_plus_non_agri_basin_stats['largest_patch_size_ha'] = existing_forest_plus_non_agri_basin_stats['largest_patch_size_ha'].fillna(0)
existing_forest_plus_non_agri_basin_stats['total_forest_area_ha'] = existing_forest_plus_non_agri_basin_stats['total_forest_area_ha'].fillna(0)

#### Calculate the percentage of forest cover in the catchment

In [ ]:
# basin_stats['percentage_forest_in_catchment'] = (basin_stats['total_forest_area'] / basin_stats['catchment_area']) * 100

# --- Calculate percentage ---
# Make sure to use values in m² (both total_forest_area and catchment_area are in m²)
existing_forest_plus_non_agri_basin_stats['pct_forest'] = (existing_forest_plus_non_agri_basin_stats['total_forest_area'] / existing_forest_plus_non_agri_basin_stats['catchment_area']) * 100
existing_forest_plus_non_agri_basin_stats['pct_forest'] = existing_forest_plus_non_agri_basin_stats['pct_forest'].fillna(0)  # set NaN to 0% for catchments with no forest

In [ ]:
# --- Add catchment area in hectares ---
existing_forest_plus_non_agri_basin_stats['catchment_area_ha'] = existing_forest_plus_non_agri_basin_stats['catchment_area'] / 10000

In [ ]:
display(existing_forest_plus_non_agri_basin_stats)

In [ ]:
# Compute NN statistics on the individual forest patches in each catchment
def compute_nn_stats(group):
    # Compute centroids of the forest patches in this catchment
    centroids = group.geometry.centroid
    # If there are fewer than 2 patches, NN distances cannot be computed
    if len(centroids) < 2:
        return pd.Series({
            'min_nn_distance': np.nan,
            'mean_nn_distance': np.nan,
            'max_nn_distance': np.nan
        })
    # Create an array of (x, y) coordinates from the centroids
    coords = np.array([(pt.x, pt.y) for pt in centroids])
    
    # Build a KDTree for fast nearest neighbor search
    tree = cKDTree(coords)
    # Query for each point: k=2 returns the point itself and its nearest neighbor
    distances, indices = tree.query(coords, k=2)
    nn_distances = distances[:, 1]  # take the second column (nearest neighbor distance)
    
    # CHANGED: Round the NN distances to whole numbers and format with commas
    return pd.Series({
        'min_nn_distance': nn_distances.min(),
        'mean_nn_distance': nn_distances.mean(),
        'max_nn_distance': nn_distances.max()
    })

# First, apply the function to compute nn_stats
nn_stats = existing_forest_and_afforestable_non_agri_in_basins.groupby("HYBAS_ID").apply(compute_nn_stats).reset_index()

# Now, rename the columns as requested
nn_stats = nn_stats.rename(columns={
    "min_nn_distance": "Minimum nearest neighbour distance (m)",
    "mean_nn_distance": "Mean nearest neighbour distance (m)",
    "max_nn_distance": "Maximum nearest neighbour distance (m)",
})

# Inspect the resulting NN statistics
display(nn_stats.head())

In [ ]:
# Merge the NN statistics with the existing basin stats on HYBAS_ID
final_stats = existing_forest_plus_non_agri_basin_stats.merge(nn_stats, on="HYBAS_ID", how="left")

# CHANGED/ADDED: Drop columns with "(m2)" in the column name, keeping only the (ha) columns (and others)
cols_to_drop = [col for col in final_stats.columns if "(m2)" in col]
final_stats_reduced = final_stats.drop(columns=cols_to_drop)

# Optionally, inspect the resulting columns
display("Final columns to be exported:")
display(final_stats_reduced.columns)

# # Save the merged DataFrame as a CSV in your output folder
# output_csv_path = output_folder / "final_basin_stats.csv"
# final_stats.to_csv(output_csv_path, index=False)

# Optionally, inspect the final results
display(final_stats_reduced.head())

In [ ]:
# Define a dictionary for the metrics you want to map.
# Now each metric also includes a "units" field.
metrics = {
    "patch_count": {
        "title": "Existing forest plus afforestable non-agricultural lands: top 5 catchments by patch count",
        "filename": "existing_forest_plus_afforestable_non_agri_top5_by_patch_count.png",
        "units": "count",
        "legend_title": "Patch count"
    },
    "total_forest_area": {
        "title": "Existing forest plus afforestable non-agricultural lands: top 5 catchments by total forest area (m²)",
        "filename": "existing_forest_plus_afforestable_non_agri_top5_by_total_forest_area.png",
        "units": "m²",
        "legend_title": "Total forest area"        
    },
    "pct_forest": {
        "title": "Existing forest plus afforestable non-agricultural lands: top 5 catchments by percentage forest",
        "filename": "existing_forest_plus_afforestable_non_agri_top5_by_pct_forest.png",
        "units": "%",
        "legend_title": "Percentage forest"
    },
    "Maximum nearest neighbour distance (m)": {
        "title": "Existing forest plus afforestable non-agricultural lands: top 5 catchments by maximum NN distance (m)",
        "filename": "existing_forest_plus_afforestable_non_agri_top5_by_max_nn_distance.png",
        "units": "m",
        "legend_title": "Maximum nearest neighbour distance"        
    },
    "total_edge_length": {
        "title": "Existing forest plus afforestable non-agricultural lands: top 5 catchments by total edge length(m)",
        "filename": "existing_forest_plus_afforestable_non_agri_top5_by_total_edge_length.png",
        "units": "m",
        "legend_title": "Total edge length"
    }
}

# Define scale bar and north arrow functions (same as your formatting)
def add_scale_bar(ax, length_km=20, location=(0.9, 0.79), linewidth=2, tick_height=0.01, label_offset=0.04, km_offset=0.01):
    x, y = location
    bar_half_length = 0.05
    ax.plot([x - bar_half_length, x + bar_half_length], [y, y],
            transform=ax.transAxes, color="black", linewidth=linewidth)
    for pos in [x - bar_half_length, x, x + bar_half_length]:
        ax.plot([pos, pos], [y - tick_height/2, y + tick_height/2],
                transform=ax.transAxes, color="black", linewidth=linewidth)
    ax.text(x - bar_half_length, y - tick_height - label_offset, "0",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x, y - tick_height - label_offset, f"{int(length_km // 2)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length, y - tick_height - label_offset, f"{int(length_km)}",
            transform=ax.transAxes, ha="center", va="center", fontsize=10)
    ax.text(x + bar_half_length + km_offset, y, "km",
            transform=ax.transAxes, ha="left", va="center", fontsize=12)

def add_north_arrow(ax, location=(0.9, 0.85), size=0.05, fontsize=12, label_offset=0.03):
    x, y = location
    ax.annotate("", xy=(x, y + size), xycoords="axes fraction",
                xytext=(x, y), textcoords="axes fraction",
                arrowprops=dict(facecolor="black", edgecolor="black", headwidth=10, headlength=15, width=5))
    ax.text(x, y + size + label_offset, "N",
            transform=ax.transAxes, fontsize=fontsize, fontweight="bold", ha="center", va="center", color="black")

# Loop over each metric and generate the corresponding map
for col, info in metrics.items():
    # Select the top 5 catchments based on the metric (using final_stats as the source)
    top5 = final_stats.nlargest(5, col)

    # Create figure and axis
    fig, ax = plt.subplots(figsize=(18, 14), dpi=300)

    # Plot background catchments and boundary for context
    hydrobasins.plot(ax=ax, color="white", edgecolor="blue")
    jamaica_boundary.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.2, zorder=100)

    # Set up colormap using Greens, so that the darkest green indicates the highest value.
    cmap = plt.get_cmap("Greens")
    norm = mcolors.Normalize(vmin=top5[col].min(), vmax=top5[col].max())

    # Plot each catchment in the top5 with annotation and prepare legend entries
    legend_handles = []
    for idx, row in top5.iterrows():
        value = row[col]
        color = cmap(norm(value))
        # Plot the polygon for the catchment
        gpd.GeoSeries(row['geometry']).plot(ax=ax, color=color, edgecolor="black", linewidth=1.5, zorder=101)
        # Determine the centroid for annotation
        centroid = row['geometry'].centroid
        # Include the units in the annotation if desired
        ax.annotate(f"{row['HYBAS_ID']}\n{value:,.2f} {info['units']}", 
                    xy=(centroid.x, centroid.y), ha="center", fontsize=10, fontweight="bold")
        patch = mpatches.Patch(color=color, label=f"{row['HYBAS_ID']} ({value:,.2f} {info['units']})")
        legend_handles.append(patch)

    # Add legend below the map
    ax.legend(handles=legend_handles, title=f"HYBAS ID and {info['legend_title']}",
              bbox_to_anchor=(0.5, -0.1), loc="upper center", ncol=3,
              frameon=False, fontsize=12, title_fontsize=14)

    # Add scale bar and north arrow
    add_scale_bar(ax, length_km=20, location=(0.9, 0.79))
    add_north_arrow(ax, location=(0.9, 0.85))

    # Set common x and y limits (assumes common_xlim and common_ylim are defined)
    ax.set_xlim(common_xlim)
    ax.set_ylim(common_ylim)
    ax.set_xlabel("Easting", fontsize=14, fontname="Times New Roman")
    ax.set_ylabel("Northing", fontsize=14, fontname="Times New Roman")
    ax.set_title(info["title"], fontsize=20, fontweight="bold", fontname="Times New Roman", pad=20)

    plt.tight_layout()
    # Save the figure using the updated filename from the metrics dictionary
    fig.savefig(map_forest_connectivity_figures_folder / info["filename"], dpi=300, bbox_inches="tight")
    plt.show()